# Vietnamese ETF Exploratory Data Analysis

This notebook analyzes historical daily OHLCV data for the five supported Vietnamese ETFs in the ETF Decision Support System. It reads from PostgreSQL using the existing backend configuration and does not modify database data.

Scope:

- E1VFVN30
- FUEVFVND
- FUEVN100
- FUEDCMID
- FUESSVFL

The analysis is exploratory only. It does not implement machine learning or BUY/HOLD/SELL recommendations.

## 1. Setup

Import the required libraries and load the database connection from the existing backend settings. Database credentials are read from environment variables or `backend/.env`; they are not hard-coded in the notebook.

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
BACKEND_ROOT = PROJECT_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.core.config import settings

plt.style.use("default")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

SUPPORTED_SYMBOLS = ["E1VFVN30", "FUEVFVND", "FUEVN100", "FUEDCMID", "FUESSVFL"]
TRADING_DAYS_PER_YEAR = 252

engine = create_engine(settings.database_url, pool_pre_ping=True)

## 2. Load Historical Prices

Load daily OHLCV records from the `etfs` and `price_history` tables. The query is read-only and filters to the five ETF symbols in scope.

In [3]:
query = text(
    """
    SELECT
        e.symbol,
        e.name,
        ph.date,
        ph.open,
        ph.high,
        ph.low,
        ph.close,
        ph.volume,
        ph.data_source
    FROM price_history AS ph
    JOIN etfs AS e ON e.id = ph.etf_id
    ORDER BY e.symbol, ph.date
    """
)

with engine.connect() as connection:
    price_df = pd.read_sql(query, connection)

price_df = price_df.loc[price_df["symbol"].isin(SUPPORTED_SYMBOLS)].copy()

price_df["date"] = pd.to_datetime(price_df["date"])
numeric_columns = ["open", "high", "low", "close", "volume"]
for column in numeric_columns:
    price_df[column] = pd.to_numeric(price_df[column], errors="coerce")

price_df = price_df.sort_values(["symbol", "date"]).reset_index(drop=True)
price_df.head()

OperationalError: (psycopg.OperationalError) connection failed: connection to server at "127.0.0.1", port 5432 failed: FATAL:  password authentication failed for user "etf_user"
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 3. Data Coverage

Check the number of records and available date range for each ETF. This confirms whether all ETFs have comparable historical coverage and highlights later inception dates.

In [ ]:
coverage = (
    price_df.groupby("symbol")
    .agg(
        row_count=("date", "size"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        data_sources=("data_source", lambda values: ", ".join(sorted(values.unique()))),
    )
    .reset_index()
)

coverage

## 4. Basic Data Quality Checks

Before calculating returns, check for common data issues: missing OHLCV values, duplicate dates per ETF, invalid OHLC relationships, negative volume, and records before the initial analysis start date.

In [ ]:
quality_rows = []
for symbol, group in price_df.groupby("symbol"):
    invalid_ohlc = (
        (group["high"] < group["low"])
        | (group["open"] > group["high"])
        | (group["open"] < group["low"])
        | (group["close"] > group["high"])
        | (group["close"] < group["low"])
    )
    quality_rows.append(
        {
            "symbol": symbol,
            "missing_ohlcv_values": int(group[numeric_columns].isna().sum().sum()),
            "duplicate_dates": int(group["date"].duplicated().sum()),
            "invalid_ohlc_rows": int(invalid_ohlc.sum()),
            "negative_volume_rows": int((group["volume"] < 0).sum()),
            "rows_before_2022_01_01": int((group["date"] < pd.Timestamp("2022-01-01")).sum()),
        }
    )

quality_summary = pd.DataFrame(quality_rows)
quality_summary

## 5. Descriptive Statistics

Summarize the distribution of OHLCV values for each ETF. This helps identify scale differences, volume differences, and unusual ranges.

In [ ]:
descriptive_stats = price_df.groupby("symbol")[["open", "high", "low", "close", "volume"]].describe()
descriptive_stats

## 6. Return, Volatility, and Drawdown Features

Calculate daily return, cumulative return, rolling 20-day annualized volatility, and drawdown. These are descriptive analytics only and are not recommendations.

In [ ]:
analysis_frames = []

for symbol, group in price_df.groupby("symbol"):
    group = group.sort_values("date").copy()
    group["daily_return"] = group["close"].pct_change()
    group["cumulative_return"] = (1 + group["daily_return"].fillna(0)).cumprod() - 1
    group["normalized_close"] = group["close"] / group["close"].iloc[0]
    group["rolling_volatility_20"] = (
        group["daily_return"].rolling(window=20).std() * np.sqrt(TRADING_DAYS_PER_YEAR)
    )
    group["wealth_index"] = 1 + group["cumulative_return"]
    group["previous_peak"] = group["wealth_index"].cummax()
    group["drawdown"] = group["wealth_index"] / group["previous_peak"] - 1
    analysis_frames.append(group)

analysis_df = pd.concat(analysis_frames, ignore_index=True)
analysis_df.head()

## 7. ETF Comparison Metrics

Compare ETFs by total return, annualized volatility, and maximum drawdown over the available data period.

In [ ]:
comparison_rows = []
for symbol, group in analysis_df.groupby("symbol"):
    group = group.sort_values("date")
    comparison_rows.append(
        {
            "symbol": symbol,
            "start_date": group["date"].min(),
            "end_date": group["date"].max(),
            "total_return": group["close"].iloc[-1] / group["close"].iloc[0] - 1,
            "annualized_volatility": group["daily_return"].std() * np.sqrt(TRADING_DAYS_PER_YEAR),
            "max_drawdown": group["drawdown"].min(),
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values("total_return", ascending=False)
comparison

## 8. Correlation Matrix of Daily Returns

Calculate the correlation matrix using daily percentage returns. This helps show how similarly the ETFs move over overlapping trading dates.

In [ ]:
return_matrix = analysis_df.pivot(index="date", columns="symbol", values="daily_return")
correlation_matrix = return_matrix.corr()
correlation_matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation_matrix, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlation_matrix.index)))
ax.set_yticklabels(correlation_matrix.index)

for row_index in range(len(correlation_matrix.index)):
    for column_index in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[row_index, column_index]
        ax.text(column_index, row_index, f"{value:.2f}", ha="center", va="center", color="black")

ax.set_title("Daily Return Correlation Matrix")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 9. Normalized Price Performance

Normalize each ETF's closing price to 1.0 at its first available observation. This makes performance comparable even when ETFs have different price levels.

In [ ]:
fig, ax = plt.subplots()
for symbol, group in analysis_df.groupby("symbol"):
    ax.plot(group["date"], group["normalized_close"], label=symbol)

ax.set_title("Normalized ETF Price Performance")
ax.set_xlabel("Date")
ax.set_ylabel("Normalized close price")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Cumulative Return

Plot cumulative return based on daily close-to-close percentage returns.

In [ ]:
fig, ax = plt.subplots()
for symbol, group in analysis_df.groupby("symbol"):
    ax.plot(group["date"], group["cumulative_return"], label=symbol)

ax.set_title("Cumulative Return")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative return")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Rolling 20-Day Volatility

Plot rolling 20-day annualized volatility. This shows how short-term risk changes over time.

In [ ]:
fig, ax = plt.subplots()
for symbol, group in analysis_df.groupby("symbol"):
    ax.plot(group["date"], group["rolling_volatility_20"], label=symbol)

ax.set_title("Rolling 20-Day Annualized Volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized volatility")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Drawdown

Drawdown measures the decline from the previous cumulative return peak. It is useful for understanding downside risk during difficult market periods.

In [ ]:
fig, ax = plt.subplots()
for symbol, group in analysis_df.groupby("symbol"):
    ax.plot(group["date"], group["drawdown"], label=symbol)

ax.set_title("ETF Drawdown")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

## 13. Notes for Graduation Report

This EDA supports the decision-support system by documenting the historical data coverage, return behavior, volatility, drawdown risk, and return correlations across the Vietnamese ETF universe. These outputs can later inform technical indicators and machine-learning feature engineering, but no predictive modeling or final recommendation logic is implemented in this notebook.